# Notebook 1 — Semantic Model Generation

## Objective

Generate artifact-specific candidate semantic models from heterogeneous
organizational artifacts to support subsequent **TBox Bootstrapping**.

This notebook implements the first implementation activity of
**Semantic Onboarding**. Each selected organizational artifact is processed
independently and transformed into a candidate semantic representation
expressed in Prolog.

At this stage, **no frozen vocabulary is imposed** and **no foundational
ontology is projected**. The generated models are candidate representations
from which a reference semantic model can subsequently be selected for
Notebook 2 — TBox Bootstrapping.

## Inputs

- Organizational artifacts represented as UTF-8 text files.
- In the thesis case study, these files correspond to textual
  representations of service-contract artifacts.

## Outputs

- Artifact-specific candidate semantic models in Prolog.
- Candidate predicates, fact categories, metadata structures, clauses,
  and factual statements extracted from each artifact.

## Main Activities

1. Artifact selection and loading.
2. Semantic interpretation of each artifact.
3. Candidate factual structure identification.
4. Artifact-specific Prolog model generation.

## Relation to the Methodology

The progression implemented here is:

`Organizational Artifacts → Candidate Semantic Models → TBox Bootstrapping`

The outputs of this notebook are **not** the TBox, ABox, or Formal Knowledge
Model. They are artifact-specific candidate semantic models used to support
the controlled vocabulary construction performed in Notebook 2.

## 1. Environment Setup

The reference implementation is designed to run in Google Colab and uses
Google Drive for input and output persistence. The Gemini API key is read
from Colab Secrets and is never stored in the notebook.

The notebook uses the current `google-genai` Python SDK. If reproducing a
specific experiment, set `MODEL_NAME` to the exact Gemini model version used
in that experiment.

In [ ]:
# Install the current Google Gen AI SDK when running in Colab.
%pip install -q google-genai

In [ ]:
# Mount Google Drive to access input artifacts and persist generated models.
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# Import required libraries.
from pathlib import Path
import re

from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
# Retrieve the Gemini API key from Google Colab Secrets.
api_key = userdata.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY was not found in Colab Secrets. "
        "Add the key before running this notebook."
    )

client = genai.Client(api_key=api_key)

## 2. Configuration

Configure the input directory, output directory, Gemini model, and artifacts
to process.

If `FILES_TO_PROCESS` is empty, all `.txt` files in `INPUT_DIRECTORY` are
processed. Otherwise, only the explicitly listed files are used.

In [ ]:
# Input and output locations.
INPUT_DIRECTORY = Path(
    "/content/drive/My Drive/KOA/contratos/contratos_gseg"
)
OUTPUT_DIRECTORY = Path(
    "/content/drive/My Drive/KOA/onboarding/contratos_gseg"
)

# Reference model used by the current notebook.
MODEL_NAME = "gemini-2.5-flash"

# Leave empty to process every .txt artifact in INPUT_DIRECTORY.
FILES_TO_PROCESS = [
    "ET_295_2024.txt",
]

# Maximum number of continuation calls when the model reaches an output limit.
MAX_GENERATION_CALLS = 6

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

In [ ]:
def selected_artifacts(input_directory: Path, files_to_process: list[str]) -> list[Path]:
    """Return the artifacts selected for semantic model generation."""
    if files_to_process:
        return [input_directory / name for name in sorted(files_to_process)]
    return sorted(input_directory.glob("*.txt"))


artifacts = selected_artifacts(INPUT_DIRECTORY, FILES_TO_PROCESS)

print(f"Selected artifacts: {len(artifacts)}")
for artifact in artifacts:
    print(f" - {artifact.name}")

## 3. Candidate Semantic Model Generation

The following component generates one candidate Prolog model for each
organizational artifact.

The prompt deliberately constrains the LLM to **document-centric semantic
structuring**. It does not perform ontological grounding, decision reasoning,
Semantic Bridging, or rule generation.

For the service-contract case study, the generated representation organizes:

- contract identification and metadata;
- contractual clauses and annexes;
- atomic factual statements explicitly supported by the artifact; and
- evidence references preserving traceability to the source text.

Predicate names are standardized in English, while artifact-derived values
remain in Portuguese. This separation keeps the structural vocabulary stable
while preserving the language of the source evidence.

In [ ]:
CONTINUE_MARKER = "%% CONTINUE"


def clean_prolog_text(text: str) -> str:
    """Remove Markdown code fences if they are returned by the model."""
    if not text:
        return ""
    text = re.sub(r"```(?:prolog|pl)?\\s*", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "")
    return text.strip()


def strip_continue_marker(text: str) -> tuple[str, bool]:
    """Remove the continuation marker and report whether it was present."""
    if not text:
        return "", False
    marker_found = CONTINUE_MARKER in text
    return text.replace(CONTINUE_MARKER, ""), marker_found


def tail_lines(text: str, n: int = 80) -> str:
    """Return the last n lines of generated content for continuation context."""
    lines = text.splitlines()
    return "\n".join(lines[-n:]) if len(lines) > n else text


def merge_by_overlap(previous: str, new: str, max_overlap_lines: int = 250) -> str:
    """Merge continuation outputs while avoiding duplicated overlapping lines."""
    previous = clean_prolog_text(previous)
    new = clean_prolog_text(new)

    if not previous:
        return (new.strip() + "\n") if new else ""
    if not new:
        return (previous.strip() + "\n") if previous else ""

    previous_lines = previous.splitlines()
    new_lines = new.splitlines()

    max_k = min(
        len(previous_lines),
        len(new_lines),
        max_overlap_lines,
    )

    overlap = 0
    for k in range(max_k, 0, -1):
        if previous_lines[-k:] == new_lines[:k]:
            overlap = k
            break

    merged = previous_lines + new_lines[overlap:]
    return "\n".join(merged).strip() + "\n"


def get_finish_reason(response) -> str:
    """Return the Gemini finish reason in a version-tolerant form."""
    try:
        finish_reason = response.candidates[0].finish_reason
        return getattr(finish_reason, "name", str(finish_reason))
    except Exception:
        return ""

In [ ]:
class SemanticModelGenerationAgent:
        """
        Generate an artifact-specific candidate semantic model in Prolog.

        This component belongs to the Semantic Model Generation activity of
        Semantic Onboarding. It does not generate the TBox, an ABox, the
        foundational ontology, or the final Formal Knowledge Model.
        """

        def __init__(self, file_path: Path, client, model_name: str):
            self.file_path = Path(file_path)
            self.client = client
            self.model_name = model_name
            self.content = self._read_document()
            self.semantic_model = ""

        def _read_document(self) -> str:
            """Read the organizational artifact as UTF-8 text."""
            if not self.file_path.exists():
                raise FileNotFoundError(
                    f"Input artifact not found: {self.file_path}"
                )
            return self.file_path.read_text(encoding="utf-8")

        def _build_initial_prompt(self) -> str:
            """Build the document-centric semantic transformation prompt."""
            return f"""
You are a Knowledge Representation expert specializing in semantic modeling
of organizational artifacts and Prolog.

Your task is to generate an artifact-specific candidate semantic model from
the organizational artifact provided below.

This is the first implementation activity of Semantic Onboarding. The purpose
is to identify and organize candidate semantic structures that can subsequently
support TBox Bootstrapping.

At this stage:
- no frozen vocabulary is imposed;
- no foundational ontology is projected;
- no decision model is considered; and
- no Semantic Bridging or decision reasoning is performed.

Do not infer information that is not explicitly supported by the artifact.

==================================================
GLOBAL CONSTRAINTS
==================================================

1. Language:
   - Use Portuguese for strings, identifiers, atom values, and labels derived
     from the source artifact.
   - Predicate names MUST be in English, as specified below.
   - Normalize identifiers using lowercase, underscores, and ASCII characters
     only.

2. Output format:
   - Return ONLY valid Prolog code.
   - Do NOT include Markdown, code fences, explanations, or prose.
   - Every Prolog fact MUST end with a period (.).

3. Semantic scope:
   - Do NOT infer obligations, rights, risks, decision outcomes, or meanings
     that are not explicitly stated in the artifact.
   - Do NOT create inference rules.
   - Do NOT perform ontological grounding.
   - Do NOT merge or reinterpret clauses.
   - Extract and organize only information explicitly supported by the text.

==================================================
GLOBAL CONTRACT IDENTIFICATION AND METADATA
==================================================

1. Define exactly one contract identifier:
   contract(contract_id).

2. Extract identification and header information appearing outside the
   contractual clauses, including when available:
   - contract number;
   - SAP / OCS identifiers;
   - contracting parties;
   - dates;
   - contractual object;
   - global monetary values; and
   - other explicit contract-level metadata.

3. Represent contract-level metadata only as:
   contract_metadata(contract_id, key, value).

4. Contract-level metadata MUST NOT reference a contractual clause unless the
   information exists only within that clause.

==================================================
CONTRACTUAL CLAUSE STRUCTURE
==================================================

1. Represent every contractual clause as:
   contract_clause(contract_id, clause_id, clause_title, clause_text).

2. Clause identifiers:
   - derive them from the clause title;
   - normalize them using lowercase, underscores, and ASCII characters;
   - keep the identifier stable and deterministic whenever the same title is
     encountered.

   Example:
   "Cláusula Terceira – Prazo de Vigência"
   -> clausula_terceira_prazo_vigencia

3. Clause text:
   - preserve the full literal text of the clause;
   - enclose textual values in single quotes;
   - escape characters when required for valid Prolog syntax.

==================================================
ATOMIC CLAUSE FACT EXTRACTION
==================================================

1. Extract atomic factual statements from each clause.

2. Represent each atomic statement as:
   contract_clause_fact(
       contract_id,
       clause_id,
       fact_type,
       fact_data,
       evidence_ref
   ).

3. Definitions:
   - fact_type:
       a concise normalized atom describing the explicit information type,
       such as payment_term, penalty, prazo, indice_reajuste, valor, condicao.

   - fact_data:
       a structured Prolog term or list containing only data explicitly stated
       in the source text. Do not add interpretation or explanation.

   - evidence_ref:
       a concise literal excerpt or stable textual reference identifying the
       supporting evidence in the corresponding clause. Do not invent evidence.

4. Each contract_clause_fact MUST:
   - refer to exactly one clause;
   - represent exactly one factual assertion;
   - remain short and structurally reusable across artifacts; and
   - remain traceable to explicit source evidence.

==================================================
ANNEXES
==================================================

1. If annexes are present, represent each annex as:
   contract_annex(contract_id, annex_id, annex_title, annex_text).

2. Extract atomic annex facts as:
   contract_annex_fact(
       contract_id,
       annex_id,
       fact_type,
       fact_data,
       evidence_ref
   ).

3. Apply the same non-inference and evidence-traceability constraints used for
   contractual clauses.

==================================================
COMPLETENESS AND OUTPUT-LIMIT HANDLING
==================================================

1. Do not intentionally omit any clause or annex present in the source text.

2. If the complete Prolog model cannot be produced because of the response
   length limit, end the response with EXACTLY this line:
   {CONTINUE_MARKER}

3. If the model is complete, do NOT output {CONTINUE_MARKER}.

==================================================
SOURCE ARTIFACT
==================================================

Source file: {self.file_path.name}

---
{self.content}
---

Return only the candidate Prolog semantic model.
""".strip()

        def _build_continuation_prompt(self, generated_text: str) -> str:
            """Build a continuation prompt after a truncated generation."""
            tail = tail_lines(clean_prolog_text(generated_text), n=80)

            return f"""
Continue the candidate Prolog semantic model exactly from where the previous
response stopped.

Rules:
- Do NOT repeat or rewrite previously generated content.
- Continue from the last incomplete predicate, term, clause, or annex.
- Preserve the same predicate vocabulary and formatting already used.
- Do NOT introduce inference rules or new interpretations.
- Return ONLY valid Prolog code.
- Do NOT use Markdown, code fences, comments, or explanations.
- If the response limit is reached again, end with exactly:
  {CONTINUE_MARKER}

Last generated lines, provided only as continuation context:
---
{tail}
---

Continue now:
""".strip()

        def _generate_semantic_model_with_gemini(self) -> str:
            """Generate the complete candidate semantic model with continuation handling."""
            generated = ""
            prompt = self._build_initial_prompt()

            config = types.GenerateContentConfig(
                temperature=0.2,
                max_output_tokens=8192,
            )

            for call_number in range(1, MAX_GENERATION_CALLS + 1):
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=config,
                )

                part = response.text or ""
                part_without_marker, has_marker = strip_continue_marker(part)

                finish_reason = get_finish_reason(response)
                hit_output_limit = "MAX_TOKENS" in finish_reason.upper()
                needs_continuation = has_marker or hit_output_limit

                generated = merge_by_overlap(
                    generated,
                    part_without_marker,
                )

                print(
                    f"Generation call {call_number}: "
                    f"finish_reason={finish_reason or 'UNKNOWN'}, "
                    f"continuation={needs_continuation}"
                )

                if not needs_continuation:
                    break

                prompt = self._build_continuation_prompt(generated)
            else:
                print(
                    "Warning: maximum number of generation calls reached. "
                    "Inspect the generated model for completeness."
                )

            return clean_prolog_text(generated).rstrip() + "\n"

        def generate_semantic_model(self) -> str:
            """Public entry point for candidate semantic model generation."""
            self.semantic_model = self._generate_semantic_model_with_gemini()
            return self.semantic_model

## 4. Batch Execution

Each selected source artifact is processed independently. The resulting
candidate model is saved using the naming convention:

`KOA_<source_artifact>.pl`

The output directory is created automatically when necessary.

In [ ]:
generated_files = []

for artifact_path in artifacts:
    print("\n" + "=" * 72)
    print(f"Processing artifact: {artifact_path.name}")

    if not artifact_path.exists():
        print(f"Skipping missing artifact: {artifact_path}")
        continue

    generator = SemanticModelGenerationAgent(
        file_path=artifact_path,
        client=client,
        model_name=MODEL_NAME,
    )

    candidate_model = generator.generate_semantic_model()

    output_name = f"KOA_{artifact_path.stem.replace(' ', '_')}.pl"
    output_path = OUTPUT_DIRECTORY / output_name
    output_path.write_text(candidate_model, encoding="utf-8")

    generated_files.append(output_path)
    print(f"Candidate semantic model saved to: {output_path}")

In [ ]:
print("\nGenerated candidate semantic models:")
for generated_file in generated_files:
    print(f" - {generated_file}")

## 5. Output and Next Step

The generated `.pl` files are **artifact-specific candidate semantic
models**. They preserve explicit source content in a normalized factual
structure but have not yet been constrained by a frozen vocabulary or
projected onto the adopted service-contract ontology.

The next implementation stage is:

**Notebook 2 — TBox Bootstrapping**

A reference semantic model is selected from the candidate models generated
here. Its predicates, metadata structures, and fact categories are then used
to establish the controlled TBox that constrains subsequent ABox generation.